# LDA Analyses (PFC, mFC dataset)

Port of `code/LEC_lda_analyses.ipynb` to the mPFC dataset under
`mFC_data/data/`. Same three LDA pipelines as LEC:

- State separation on trial-bins ([lda_state_analysis_trialbins.py](lda_state_analysis_trialbins.py)) — including Pillai trace, LOTO / within-task / within-session-shuffle controls, and pseudotime-matched correlations.
- Goal-progress LDA ([lda_state_analysis_goalprogress.py](lda_state_analysis_goalprogress.py)).
- Reward × goal-progress LDA ([lda_reward_goalprogress.py](lda_reward_goalprogress.py)).

Only the data-loading cells are PFC-specific. The reward × goal-progress
section needs the pre-computed `Neurons_norm` field, so the PFC loader is
called with `compute_norm=True`.

## Method notes

**Trial extraction (state separation)**: each session's raw recording is z-scored across the concatenated timeline, then for every (trial, state) the mean population vector is taken across the state's raw time window. The result is a matrix $X \in \mathbb{R}^{N \times n_{neurons}}$ labelled by state $\in \{A,B,C,D\}$.

**Pillai's trace**: the sum $\sum_i \lambda_i / (1 + \lambda_i)$ over eigenvalues of $W^{-1}B$ where $W$ is the within-class scatter and $B$ is the between-class scatter. Bounded in $[0, n_{classes} - 1]$; equals 0 when classes are inseparable, approaches $n_{classes} - 1$ when perfectly separable.

**Roll-shuffle null**: each session's raw recording is circularly rolled by a random offset (at least one mean trial length) before re-running the full pipeline. This preserves autocorrelation structure while destroying state-activity alignment.

**Trial balancing** (new flag): if `balance_trials_per_task=True`, each session's trials are subsampled to `min(n_trials)` across sessions *before* trial-state extraction. The z-score still uses the full raw recording for stable normalisation. Prevents tasks with more trials from dominating the within-state scatter matrix. Applied identically to real and shuffled data.

In [1]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
import os, pickle
from importlib import reload

In [2]:
DATA_FOLDER = '/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data'
META = os.path.join(DATA_FOLDER, 'MetaData')

In [3]:
mouse_recdays = list(np.load(os.path.join(META, 'combined_ABCDonly_days.npy')).astype(str))
print(f'{len(mouse_recdays)} recdays in combined_ABCDonly_days.npy')
print('first 3:', mouse_recdays[:3])

25 recdays in combined_ABCDonly_days.npy
first 3: [np.str_('ab03_01092023_02092023'), np.str_('ab03_05092023_06092023'), np.str_('ab03_29082023_30082023')]


In [ ]:
# compute_norm=True populates Neurons_norm + Locs_norm needed by the reward × goal-progress
# LDA section. The trialbins / goalprogress LDAs only need raw data and work either way.
# Memory note: this is ~4 GB across all 25 recdays. For a quick first pass, uncomment
# the next line to slice down.
# mouse_recdays = mouse_recdays[:5]

from glm_analysis_v2 import build_data_dic_from_pfc
data_dic = build_data_dic_from_pfc(DATA_FOLDER, mouse_recdays, compute_norm=True)
mouse_recdays = sorted(data_dic.keys())   # in case any were skipped
print(f'\n{len(mouse_recdays)} recdays loaded')

  ab03_01092023_02092023: 7 sessions
  ab03_05092023_06092023: 8 sessions
  ab03_29082023_30082023: 9 sessions
  ah03_12082021_13082021: 8 sessions
  ah03_18082021_19082021: 8 sessions
  ah04_01122021_02122021: 8 sessions
  ah04_05122021_06122021: 8 sessions
  ah04_07122021_08122021: 8 sessions


In [ ]:
# Build valid_sessions_dic: for each mouse_recday, keep one session per unique task structure
# Identical logic to the LEC notebook; works on PFC because num_trials and Task are present.
valid_sessions_dic = {}
for mouse_recday in mouse_recdays:
    valid_sessions = []
    tasks = []
    for session in list(data_dic[mouse_recday].keys()):
        if session == 'valid_sessions':
            continue
        if data_dic[mouse_recday][session]['num_trials'] < 5:
            print(f'{mouse_recday} session {session}: not enough trials, skipping')
            continue
        if 'defaultdict' in str(data_dic[mouse_recday][session]['Task']):
            print(f'{mouse_recday} session {session}: no task, skipping')
            continue
        if not any(np.array_equal(data_dic[mouse_recday][session]['Task'], candidate)
                   for candidate in tasks):
            tasks.append(data_dic[mouse_recday][session]['Task'])
            valid_sessions.append(session)
    print(f'{mouse_recday}: {valid_sessions}')
    valid_sessions_dic[mouse_recday] = valid_sessions

## State separation LDA (trial-bins)

In [ ]:
import lda_state_analysis_trialbins
import lda_state_separation
reload(lda_state_analysis_trialbins)
reload(lda_state_separation)
from lda_state_analysis_trialbins import run_trialbins_lda_analysis, plot_ld_projections
from lda_state_analysis_trialbins import EVENT_LABELS, EVENT_COLOURS

mouse_recday = mouse_recdays[0]

results_unbalanced = run_trialbins_lda_analysis(
    data_dic, mouse_recday,
    valid_sessions=valid_sessions_dic[mouse_recday],
    neuron_subset=None,
    balance_trials_per_task=False,
)
plot_ld_projections(results_unbalanced['X_state_ld'], results_unbalanced['y_state'],
                    EVENT_LABELS, EVENT_COLOURS, f'{mouse_recday} (unbalanced)')

## Trial-balanced state separation

Same analysis as above, but with `balance_trials_per_task=True` — each session's trials are subsampled to `min(n_trials)` across sessions before (trial, state) vector extraction. Sample count drops accordingly; class balance per state is preserved across tasks.

In [ ]:
results_balanced = run_trialbins_lda_analysis(
    data_dic, mouse_recday,
    valid_sessions=valid_sessions_dic[mouse_recday],
    neuron_subset=None,
    balance_trials_per_task=True,
    balance_rng_seed=42,
)
plot_ld_projections(results_balanced['X_state_ld'], results_balanced['y_state'],
                    EVENT_LABELS, EVENT_COLOURS, f'{mouse_recday} (balanced)')

print(f"Unbalanced samples: {results_unbalanced['X'].shape[0]}")
print(f"Balanced samples:   {results_balanced['X'].shape[0]}")
print(f"Unbalanced LOGO CV acc: {results_unbalanced['real_acc']:.3f} (p={results_unbalanced['p_value']:.4f})")
print(f"Balanced LOGO CV acc:   {results_balanced['real_acc']:.3f} (p={results_balanced['p_value']:.4f})")

## Pillai-trace separability (trial-balanced)

Fits the same 4-class LDA but reports Pillai's trace and the per-LD eigenvalues, with a roll-shuffle null on the raw recordings. Trial balancing is on by default here.

In [ ]:
from lda_state_analysis_trialbins import run_lda_eigenvalue_separation

for mouse_recday in mouse_recdays:
    sep = run_lda_eigenvalue_separation(
        data_dic, mouse_recday,
        valid_sessions=valid_sessions_dic[mouse_recday],
        neuron_subset=None,
        n_shuffles=100,
        balance_trials_per_task=True,
    )
    print(f"Pillai={sep['pillai']:.4f}, p={sep['p_value']:.4f}")
    print(f"Eigenvalues: {np.round(sep['eigenvalues'], 3)}")

### Controls for the Pillai-trace analysis

Three complementary tests of whether the large real-vs-null Pillai gap above reflects genuine **cross-task state tuning** vs an artefact of within-task similarity:

1. **T2 (within-task Pillai)** — fit LDA within each session separately and compare the average within-task Pillai to the across-task Pillai. If across-task ≫ within-task, pooling tasks adds signal. If similar, the across-task LDA is just averaging within-task structure.
2. **T1 (LOTO Pillai)** — fit LDA on N−1 sessions, project the held-out session's data through that fixed LDA, compute Pillai. Tests whether the LDA *direction* transfers across tasks.
3. **T3 (within-session shuffle null)** — stricter alternative null. Shuffles state labels within each session (preserving session structure, autocorrelation, and the actual extracted vectors) instead of rolling raw data. Conservative robustness check.

In [ ]:
# T2: within-task Pillai trace per session.
# Compare mean within-task Pillai to the across-task Pillai (`sep['pillai']` from cell above).

import lda_state_analysis_trialbins
reload(lda_state_analysis_trialbins)
from lda_state_analysis_trialbins import run_within_task_pillai

within_task_results = {}
across_task_pillais = {}  # mr -> across-task Pillai (one run of the original analysis)

for mr in mouse_recdays:
    try:
        within_task_results[mr] = run_within_task_pillai(
            data_dic, mr,
            valid_sessions=valid_sessions_dic[mr],
            n_shuffles=50,
        )
    except Exception as e:
        print(f'  {mr}: skipped ({e})')

# Per-mouse_recday: mean within-task Pillai vs across-task Pillai
recdays_sorted = sorted(within_task_results.keys())
fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(recdays_sorted)), 5))
x = np.arange(len(recdays_sorted))
within_means = [within_task_results[mr]['mean_within_task_pillai'] for mr in recdays_sorted]
ax.bar(x, within_means, color='steelblue', alpha=0.75, edgecolor='black',
       label='Mean within-task Pillai (per session)')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(recdays_sorted, rotation=45, ha='right', fontsize=7)
ax.set_ylabel("Pillai's trace")
ax.set_title('T2: mean within-task Pillai per mouse_recday')
ax.legend()
plt.tight_layout()
plt.show()

# Printed summary
print('Mean within-task Pillai per mouse_recday:')
for mr in recdays_sorted:
    r = within_task_results[mr]
    per_sess = r['per_session_pillais']
    valid = ~np.isnan(per_sess)
    print(f"  {mr}: \u03bc={r['mean_within_task_pillai']:.4f}  "
          f"(per-session: {np.round(per_sess[valid], 3).tolist()})")

In [ ]:
# T1: leave-one-task-out Pillai trace.
# For each held-out session, fit LDA on the rest; project held-out through it; compute Pillai.
# Tests whether the LDA direction generalises across tasks.

import lda_state_analysis_trialbins
reload(lda_state_analysis_trialbins)
from lda_state_analysis_trialbins import run_lda_eigenvalue_separation_loto

loto_results = {}
for mr in mouse_recdays:
    try:
        loto_results[mr] = run_lda_eigenvalue_separation_loto(
            data_dic, mr,
            valid_sessions=valid_sessions_dic[mr],
            n_shuffles=50,
        )
    except Exception as e:
        print(f'  {mr}: skipped ({e})')

# Pool LOTO Pillai across mouse_recdays
recdays_sorted = sorted(loto_results.keys())
all_real = np.concatenate([loto_results[mr]['loto_pillais'] for mr in recdays_sorted])
all_real = all_real[~np.isnan(all_real)]
all_null = np.concatenate([loto_results[mr]['null_loto_pillais'].ravel() for mr in recdays_sorted])
all_null = all_null[~np.isnan(all_null)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel 1: pooled real vs null distribution
ax = axes[0]
bins = np.linspace(
    min(all_null.min(), all_real.min()) - 0.05,
    max(all_null.max(), all_real.max()) + 0.05, 41)
ax.hist(all_null, bins=bins, color='0.7', alpha=0.7, density=True, label='Null (label shuffle)')
ax.hist(all_real, bins=bins, color='crimson', alpha=0.7, density=True, label='Real LOTO Pillai')
ax.axvline(all_real.mean(), color='crimson', linewidth=2)
ax.axvline(all_null.mean(), color='0.4', linewidth=2)
ax.set_xlabel("LOTO Pillai's trace")
ax.set_ylabel('Density')
ax.set_title(f'T1: pooled LOTO Pillai\n(\u03bc_real={all_real.mean():.3f}, \u03bc_null={all_null.mean():.3f})')
ax.legend()

# Panel 2: per-mouse_recday mean LOTO Pillai vs null 95th pct
ax = axes[1]
x = np.arange(len(recdays_sorted))
mean_real = [np.nanmean(loto_results[mr]['loto_pillais']) for mr in recdays_sorted]
null_95 = [np.nanpercentile(loto_results[mr]['null_loto_pillais'], 95) for mr in recdays_sorted]
ax.bar(x, mean_real, color='crimson', alpha=0.8, edgecolor='black', label='Real (mean across folds)')
ax.plot(x, null_95, 'ko--', label='Null 95th pct')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(recdays_sorted, rotation=45, ha='right', fontsize=7)
ax.set_ylabel("LOTO Pillai's trace")
ax.set_title('Per-recday LOTO Pillai vs null')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# T3: within-session label-shuffle null.
# Stricter alternative null. Shuffles state labels within each session, preserving
# the actual extracted vectors and within-session structure. Compare to the
# existing roll-shuffle null.

import lda_state_analysis_trialbins
reload(lda_state_analysis_trialbins)
from lda_state_analysis_trialbins import run_within_session_shuffle_null

ws_null_results = {}
for mr in mouse_recdays:
    try:
        ws_null_results[mr] = run_within_session_shuffle_null(
            data_dic, mr,
            valid_sessions=valid_sessions_dic[mr],
            n_shuffles=100,
        )
    except Exception as e:
        print(f'  {mr}: skipped ({e})')

# Per-recday: real Pillai vs within-session-shuffle null 95th pct
recdays_sorted = sorted(ws_null_results.keys())
fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(recdays_sorted)), 5))
x = np.arange(len(recdays_sorted))
real_p = [ws_null_results[mr]['pillai'] for mr in recdays_sorted]
null_95 = [np.nanpercentile(ws_null_results[mr]['null_pillais'], 95) for mr in recdays_sorted]
null_50 = [np.nanpercentile(ws_null_results[mr]['null_pillais'], 50) for mr in recdays_sorted]
ax.bar(x, real_p, color='crimson', alpha=0.8, edgecolor='black', label='Real Pillai')
ax.plot(x, null_95, 'ko--', label='WS-shuffle null 95th pct')
ax.plot(x, null_50, 'kx:',  label='WS-shuffle null median')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(recdays_sorted, rotation=45, ha='right', fontsize=7)
ax.set_ylabel("Pillai's trace")
ax.set_title('T3: real Pillai vs within-session-shuffle null')
ax.legend()
plt.tight_layout()
plt.show()

# Printed p-values per recday
print('Per-recday p-values under within-session shuffle null:')
for mr in recdays_sorted:
    r = ws_null_results[mr]
    print(f"  {mr}: Pillai={r['pillai']:.4f}, p={r['p_value']:.4f}")

## Pseudotime-matched state correlation (drift control)

For each pair of sessions and each *shared trial number*, correlate the population vector of state X in task 1 with that of state Y in task 2. Matching by trial number controls for slow drift in firing rates — paired trials sit at roughly the same point in the recording timeline, so drift affects both similarly. We then bucket the resulting correlations by whether the states match (within-state: X=Y) or differ (across-state: X≠Y).

**Prediction**: if neurons carry state information beyond drift, **within-state correlations exceed across-state correlations**. The two-sample t-test in the right panel quantifies the gap; the per-recday paired test reported at the bottom is more conservative (treats each recording as a single observation).

Uses `build_trialbins_dataset` + `compute_pseudotime_state_correlations` from `lda_state_analysis_trialbins.py`.

In [ ]:
# Pseudotime-matched state correlation across mouse_recdays.
# For each (session_pair, shared trial number, state_a, state_b): correlate the
# two population vectors. Within-state pairs (state_a == state_b) should exceed
# across-state pairs if state tuning isn't explained by global drift.

reload(lda_state_analysis_trialbins)
from lda_state_analysis_trialbins import (
    build_trialbins_dataset,
    compute_pseudotime_state_correlations,
    EVENT_LABELS,
)

pooled_within = []
pooled_across = []
per_recday = {}  # mouse_recday -> (mean_within, mean_across, n_within, n_across)
per_recday_arrays = {}  # mouse_recday -> (within_arr, across_arr) for downstream grouping

for mr in mouse_recdays:
    try:
        X, y_state, sess_id, trial_id, _, _ = build_trialbins_dataset(
            data_dic, mr,
            valid_sessions=valid_sessions_dic[mr],
            neuron_subset=None,
        )
    except Exception as e:
        print(f'  {mr}: skipped ({e})')
        continue

    within, across = compute_pseudotime_state_correlations(X, y_state, sess_id, trial_id)
    if not within or not across:
        print(f'  {mr}: no comparable pairs, skipped')
        continue

    w = np.array([d['corr'] for d in within])
    a = np.array([d['corr'] for d in across])
    pooled_within.append(w)
    pooled_across.append(a)
    per_recday[mr] = (w.mean(), a.mean(), len(w), len(a))
    per_recday_arrays[mr] = (w, a)
    print(f"  {mr}: within \u03bc={w.mean():.3f} (n={len(w)})  across \u03bc={a.mean():.3f} (n={len(a)})")

pooled_within = np.concatenate(pooled_within)
pooled_across = np.concatenate(pooled_across)

# --- Cross-mouse summary figure ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bins = np.linspace(-1, 1, 51)
ax.hist(pooled_across, bins=bins, color='steelblue', alpha=0.55,
        label=f'Across state (n={len(pooled_across)}, \u03bc={pooled_across.mean():.3f})')
ax.hist(pooled_within, bins=bins, color='crimson', alpha=0.55,
        label=f'Within state (n={len(pooled_within)}, \u03bc={pooled_within.mean():.3f})')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(pooled_across.mean(), color='steelblue', linewidth=2)
ax.axvline(pooled_within.mean(), color='crimson', linewidth=2)
t_stat, p_val = st.ttest_ind(pooled_within, pooled_across)
ax.text(0.02, 0.97, f'two-sample t = {t_stat:.2f}\np = {p_val:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
ax.set_xlabel('Population vector correlation')
ax.set_ylabel('Count')
ax.set_title('Pooled within vs across state\n(trial-matched across tasks)')
ax.legend(loc='upper right')

ax = axes[1]
recdays_sorted = sorted(per_recday.keys())
w_means = [per_recday[mr][0] for mr in recdays_sorted]
a_means = [per_recday[mr][1] for mr in recdays_sorted]
x = np.arange(len(recdays_sorted))
ax.bar(x - 0.18, a_means, width=0.36, color='steelblue', label='Across state', edgecolor='black')
ax.bar(x + 0.18, w_means, width=0.36, color='crimson', label='Within state', edgecolor='black')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(recdays_sorted, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Mean correlation')
ax.set_title('Per-recday means')
ax.legend()

plt.tight_layout()
plt.show()

# Paired test at the recday level (more conservative than pooling)
diffs = np.array(w_means) - np.array(a_means)
t_paired, p_paired = st.ttest_1samp(diffs, 0)
print(f"\nPer-recday paired test (within - across): "
      f"mean \u0394={diffs.mean():.4f}, t={t_paired:.2f}, p={p_paired:.2e}")

### Per-mouse breakdown

Aggregate within/across-state correlations by **mouse identity** (the first underscore-separated token of `mouse_recday`, e.g. `ah08`). Concatenates all of a mouse's recdays and produces (a) a histogram grid (one per mouse, overlaying within vs across) and (b) a bar chart of per-mouse means.

Per-mouse paired test (treats each mouse as one observation of `within − across` averaged across its recdays) is the most conservative inference at this granularity.

In [ ]:
# Per-mouse aggregation of pseudotime correlations.
from collections import defaultdict

# Group recdays by mouse identity (first '_' token of mouse_recday)
mouse_to_recdays = defaultdict(list)
for mr in per_recday_arrays:
    mouse = mr.split('_')[0]
    mouse_to_recdays[mouse].append(mr)

mice_sorted = sorted(mouse_to_recdays.keys())

# Concatenate within/across arrays per mouse
by_mouse = {}
for mouse in mice_sorted:
    w_all = np.concatenate([per_recday_arrays[mr][0] for mr in mouse_to_recdays[mouse]])
    a_all = np.concatenate([per_recday_arrays[mr][1] for mr in mouse_to_recdays[mouse]])
    by_mouse[mouse] = (w_all, a_all)
    print(f'  {mouse}: {len(mouse_to_recdays[mouse])} recdays, '
          f'within \u03bc={w_all.mean():.3f} (n={len(w_all)})  '
          f'across \u03bc={a_all.mean():.3f} (n={len(a_all)})')

# --- Figure 1: histogram grid (one panel per mouse) ---
n_mice = len(mice_sorted)
ncols = min(n_mice, 3)
nrows = (n_mice + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
bins = np.linspace(-1, 1, 41)
for i, mouse in enumerate(mice_sorted):
    r, c = divmod(i, ncols)
    ax = axes[r, c]
    w_arr, a_arr = by_mouse[mouse]
    ax.hist(a_arr, bins=bins, color='steelblue', alpha=0.55,
            label=f'Across (\u03bc={a_arr.mean():.3f})')
    ax.hist(w_arr, bins=bins, color='crimson', alpha=0.55,
            label=f'Within (\u03bc={w_arr.mean():.3f})')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.axvline(a_arr.mean(), color='steelblue', linewidth=2)
    ax.axvline(w_arr.mean(), color='crimson', linewidth=2)
    t, p = st.ttest_ind(w_arr, a_arr)
    ax.text(0.02, 0.97, f't={t:.2f}\np={p:.2e}',
            transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.set_title(f'Mouse {mouse} ({len(mouse_to_recdays[mouse])} recdays)')
    ax.set_xlabel('Population vector correlation')
    ax.set_ylabel('Count')
    ax.legend(loc='upper right', fontsize=8)
# Hide any unused axes
for j in range(n_mice, nrows * ncols):
    r, c = divmod(j, ncols)
    axes[r, c].axis('off')
plt.suptitle('Per-mouse pseudotime-matched state correlation',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Figure 2: per-mouse bar chart of within/across means ---
fig, ax = plt.subplots(figsize=(8, 5))
w_mouse_means = [by_mouse[m][0].mean() for m in mice_sorted]
a_mouse_means = [by_mouse[m][1].mean() for m in mice_sorted]
w_mouse_sems = [by_mouse[m][0].std() / np.sqrt(len(by_mouse[m][0])) for m in mice_sorted]
a_mouse_sems = [by_mouse[m][1].std() / np.sqrt(len(by_mouse[m][1])) for m in mice_sorted]
x = np.arange(len(mice_sorted))
ax.bar(x - 0.18, a_mouse_means, yerr=a_mouse_sems, width=0.36,
       color='steelblue', label='Across state', edgecolor='black', capsize=4)
ax.bar(x + 0.18, w_mouse_means, yerr=w_mouse_sems, width=0.36,
       color='crimson', label='Within state', edgecolor='black', capsize=4)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(mice_sorted)
ax.set_ylabel('Mean correlation')
ax.set_title('Per-mouse mean within vs across state correlation')
ax.legend()
plt.tight_layout()
plt.show()

# --- Per-mouse paired test (most conservative; one observation per mouse) ---
mouse_diffs = np.array(w_mouse_means) - np.array(a_mouse_means)
if len(mouse_diffs) >= 2:
    t_m, p_m = st.ttest_1samp(mouse_diffs, 0)
    print(f'\nPer-mouse paired test (within - across): '
          f'mean \u0394={mouse_diffs.mean():.4f}, t={t_m:.2f}, p={p_m:.2e}  (n_mice={len(mouse_diffs)})')
else:
    print(f'\nOnly {len(mouse_diffs)} mouse(s); paired test not meaningful.')

#### Aggregator: mean vs max per state window

The pseudotime analysis above used **mean** firing across each state window for the per-(trial, state) population vectors. The cell below reruns the cross-task analysis with **max** instead — useful when state-tuning is transient/punctate rather than sustained. Side-by-side comparison of pooled within vs across distributions under both aggregators.

In [ ]:
# Pseudotime-matched correlation with MAX aggregation per state (vs the MEAN above).
# Uses build_trialbins_dataset(..., aggregator='max').

reload(lda_state_analysis_trialbins)
from lda_state_analysis_trialbins import (
    build_trialbins_dataset,
    compute_pseudotime_state_correlations,
)

pooled_within_max = []
pooled_across_max = []
for mr in mouse_recdays:
    try:
        X, y_state, sess_id, trial_id, _, _ = build_trialbins_dataset(
            data_dic, mr,
            valid_sessions=valid_sessions_dic[mr],
            neuron_subset=None,
            aggregator='max',
        )
    except Exception as e:
        print(f'  {mr}: skipped ({e})')
        continue
    within, across = compute_pseudotime_state_correlations(X, y_state, sess_id, trial_id)
    if within and across:
        pooled_within_max.append(np.array([d['corr'] for d in within]))
        pooled_across_max.append(np.array([d['corr'] for d in across]))

pooled_within_max = np.concatenate(pooled_within_max)
pooled_across_max = np.concatenate(pooled_across_max)

# Side-by-side: mean (re-uses pooled_within / pooled_across from cell 14) vs max
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
bins = np.linspace(-1, 1, 51)
for ax, (lbl, w, a) in zip(axes, [
        ('mean per state', pooled_within, pooled_across),
        ('max  per state', pooled_within_max, pooled_across_max)]):
    ax.hist(a, bins=bins, color='steelblue', alpha=0.55,
            label=f'Across (\u03bc={a.mean():.3f})')
    ax.hist(w, bins=bins, color='crimson', alpha=0.55,
            label=f'Within (\u03bc={w.mean():.3f})')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.axvline(a.mean(), color='steelblue', linewidth=2)
    ax.axvline(w.mean(), color='crimson', linewidth=2)
    t, p = st.ttest_ind(w, a)
    ax.text(0.02, 0.97, f't={t:.2f}\np={p:.2e}',
            transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.set_xlabel('Population vector correlation')
    ax.set_title(f'Aggregator: {lbl}')
    ax.legend(loc='upper right', fontsize=8)
axes[0].set_ylabel('Count')
plt.suptitle('Pseudotime-matched state correlation: mean vs max aggregator',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Per goal-progress bin breakdown (early / middle / late)

Split each state's 90-bin window into 3 equal sub-windows (goal-progress bins). Rerun the pseudotime-matched within-vs-across-state correlation analysis separately within each gp bin. Tests whether the within > across pattern holds across all phases of goal progress (early/middle/late within each state), or is concentrated at a particular phase (e.g., reward approach).

Uses `build_trialbins_gp_dataset` from `lda_state_analysis_trialbins.py`, which extracts one population vector per (trial, state, gp_bin). The cell below filters by gp bin and reuses `compute_pseudotime_state_correlations` for the within/across correlation logic.

In [ ]:
# Pseudotime-matched state correlation, broken down by goal-progress bin.
from lda_state_analysis_trialbins import build_trialbins_gp_dataset

GP_LABELS = ['early', 'middle', 'late']
NUM_GP_BINS = 3

pooled_by_gp = {gp: {'within': [], 'across': []} for gp in range(NUM_GP_BINS)}
per_recday_by_gp = {gp: {} for gp in range(NUM_GP_BINS)}

for mr in mouse_recdays:
    try:
        X, y_state, y_gp, sess_id, trial_id, _ = build_trialbins_gp_dataset(
            data_dic, mr,
            valid_sessions=valid_sessions_dic[mr],
            neuron_subset=None,
            num_gp_bins=NUM_GP_BINS,
        )
    except Exception as e:
        print(f'  {mr}: skipped ({e})')
        continue

    for gp in range(NUM_GP_BINS):
        mask = y_gp == gp
        if mask.sum() == 0:
            continue
        within, across = compute_pseudotime_state_correlations(
            X[mask], y_state[mask], sess_id[mask], trial_id[mask]
        )
        if not within or not across:
            continue
        w = np.array([d['corr'] for d in within])
        a = np.array([d['corr'] for d in across])
        pooled_by_gp[gp]['within'].append(w)
        pooled_by_gp[gp]['across'].append(a)
        per_recday_by_gp[gp][mr] = (w, a)

# --- Figure: pooled distributions per gp bin (1 x 3) ---
fig, axes = plt.subplots(1, NUM_GP_BINS, figsize=(5 * NUM_GP_BINS, 4.5), sharey=True)
bins = np.linspace(-1, 1, 51)
for gp, gp_label in enumerate(GP_LABELS):
    ax = axes[gp]
    if not pooled_by_gp[gp]['within']:
        ax.set_title(f'{gp_label} - no data')
        continue
    w = np.concatenate(pooled_by_gp[gp]['within'])
    a = np.concatenate(pooled_by_gp[gp]['across'])
    ax.hist(a, bins=bins, color='steelblue', alpha=0.55,
            label=f'Across (n={len(a)}, \u03bc={a.mean():.3f})')
    ax.hist(w, bins=bins, color='crimson', alpha=0.55,
            label=f'Within (n={len(w)}, \u03bc={w.mean():.3f})')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.axvline(a.mean(), color='steelblue', linewidth=2)
    ax.axvline(w.mean(), color='crimson', linewidth=2)
    t, p = st.ttest_ind(w, a)
    ax.text(0.02, 0.97, f't={t:.2f}\np={p:.2e}',
            transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.set_xlabel('Population vector correlation')
    if gp == 0:
        ax.set_ylabel('Count')
    ax.set_title(f'GP bin: {gp_label}')
    ax.legend(loc='upper right', fontsize=8)
plt.suptitle('Pseudotime-matched state correlation by goal-progress bin',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Per-mouse paired test per gp bin (most conservative inference) ---
print('\nPer-mouse paired test (within - across) by gp bin:')
for gp in range(NUM_GP_BINS):
    by_mr = per_recday_by_gp[gp]
    if not by_mr:
        print(f'  {GP_LABELS[gp]:8s}: no data')
        continue
    mouse_diffs = []
    for mouse in mice_sorted:
        recdays_m = [mr for mr in by_mr if mr.startswith(mouse + '_')]
        if not recdays_m:
            continue
        w_m = np.concatenate([by_mr[mr][0] for mr in recdays_m])
        a_m = np.concatenate([by_mr[mr][1] for mr in recdays_m])
        mouse_diffs.append(w_m.mean() - a_m.mean())
    md_arr = np.array(mouse_diffs)
    if len(md_arr) >= 2:
        t_m, p_m = st.ttest_1samp(md_arr, 0)
        print(f'  {GP_LABELS[gp]:8s}: n_mice={len(md_arr)}  \u0394={md_arr.mean():.4f}  '
              f't={t_m:.2f}  p={p_m:.2e}')
    else:
        print(f'  {GP_LABELS[gp]:8s}: only {len(md_arr)} mouse(s); skipped')

#### Per-mouse × goal-progress breakdown

Same goal-progress-binned within-vs-across-state analysis as the cell above, but broken out per mouse. Tests whether the phase-specificity of state tuning (if any) is consistent across animals or driven by a subset. Uses the `per_recday_by_gp` data already computed above (no re-fitting).

In [ ]:
# Per-mouse × goal-progress aggregation. Reuses per_recday_by_gp from the cell above
# and mice_sorted from the per-mouse cell. No re-extraction needed.

by_mouse_gp = {gp: {} for gp in range(NUM_GP_BINS)}  # {gp: {mouse: (w, a)}}
for gp in range(NUM_GP_BINS):
    for mouse in mice_sorted:
        recdays_m = [mr for mr in per_recday_by_gp[gp] if mr.startswith(mouse + '_')]
        if not recdays_m:
            continue
        w = np.concatenate([per_recday_by_gp[gp][mr][0] for mr in recdays_m])
        a = np.concatenate([per_recday_by_gp[gp][mr][1] for mr in recdays_m])
        by_mouse_gp[gp][mouse] = (w, a)

# --- Figure 1: histogram grid (rows = gp bin, cols = mouse) ---
n_mice = len(mice_sorted)
fig, axes = plt.subplots(NUM_GP_BINS, n_mice, figsize=(4 * n_mice, 3 * NUM_GP_BINS),
                         sharex=True, squeeze=False)
bins = np.linspace(-1, 1, 41)
for gp, gp_label in enumerate(GP_LABELS):
    for col, mouse in enumerate(mice_sorted):
        ax = axes[gp, col]
        if mouse not in by_mouse_gp[gp]:
            ax.set_title(f'{mouse}\n{gp_label} (no data)', fontsize=9)
            ax.axis('off')
            continue
        w, a = by_mouse_gp[gp][mouse]
        ax.hist(a, bins=bins, color='steelblue', alpha=0.55, label='Across')
        ax.hist(w, bins=bins, color='crimson', alpha=0.55, label='Within')
        ax.axvline(0, color='black', linestyle='--', linewidth=0.7)
        ax.axvline(a.mean(), color='steelblue', linewidth=1.5)
        ax.axvline(w.mean(), color='crimson', linewidth=1.5)
        t, p = st.ttest_ind(w, a)
        ax.text(0.02, 0.97,
                f'\u0394={w.mean() - a.mean():+.3f}\nt={t:.2f}\np={p:.1e}',
                transform=ax.transAxes, va='top', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
        if gp == 0:
            ax.set_title(f'{mouse}', fontsize=10)
        if col == 0:
            ax.set_ylabel(f'{gp_label}\n\nCount', fontsize=10)
        if gp == NUM_GP_BINS - 1:
            ax.set_xlabel('Pop vec correlation', fontsize=9)
        if gp == 0 and col == n_mice - 1:
            ax.legend(loc='upper right', fontsize=7)
plt.suptitle('Per-mouse × goal-progress: within vs across state correlation',
             fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

# --- Figure 2: per-mouse grouped bar chart (within/across × gp bin) ---
fig, ax = plt.subplots(figsize=(10, 5))
bar_width = 0.85 / (2 * NUM_GP_BINS)
x = np.arange(n_mice)
gp_colors_within = ['#ff9aa2', '#d62728', '#7a0c10']
gp_colors_across = ['#aec7e8', '#1f77b4', '#0e3a5e']
for gp, gp_label in enumerate(GP_LABELS):
    a_means = [by_mouse_gp[gp][m][1].mean() if m in by_mouse_gp[gp] else np.nan for m in mice_sorted]
    w_means = [by_mouse_gp[gp][m][0].mean() if m in by_mouse_gp[gp] else np.nan for m in mice_sorted]
    offset_a = (gp * 2 - NUM_GP_BINS + 0.5) * bar_width
    offset_w = (gp * 2 - NUM_GP_BINS + 1.5) * bar_width
    ax.bar(x + offset_a, a_means, width=bar_width, color=gp_colors_across[gp],
           edgecolor='black', linewidth=0.5, label=f'Across, {gp_label}')
    ax.bar(x + offset_w, w_means, width=bar_width, color=gp_colors_within[gp],
           edgecolor='black', linewidth=0.5, label=f'Within, {gp_label}')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(mice_sorted)
ax.set_ylabel('Mean correlation')
ax.set_title('Per-mouse mean within vs across state correlation, by goal-progress bin')
ax.legend(ncol=3, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.show()

# --- Per-mouse Δ printout (within - across) by gp bin ---
print('Per-mouse \u0394 (within - across) by gp bin:')
print(f"  {'mouse':6s}  " + ' '.join(f'{lab:>10s}' for lab in GP_LABELS))
for mouse in mice_sorted:
    cells = []
    for gp in range(NUM_GP_BINS):
        if mouse in by_mouse_gp[gp]:
            w, a = by_mouse_gp[gp][mouse]
            cells.append(f'{w.mean() - a.mean():+.4f}')
        else:
            cells.append('     ---')
    print(f"  {mouse:6s}  " + ' '.join(f'{c:>10s}' for c in cells))

## Goal-progress LDA

Splits each (trial, state) into early / middle / late goal-progress bins and fits a separate 4-class LDA within each bin. Useful for asking whether state representation changes across the approach-to-reward trajectory.

In [ ]:
import lda_state_analysis_goalprogress
reload(lda_state_analysis_goalprogress)
from lda_state_analysis_goalprogress import run_goalprogress_lda_analysis

gp_results = run_goalprogress_lda_analysis(
    data_dic, mouse_recday,
    valid_sessions=valid_sessions_dic[mouse_recday],
    neuron_subset=None,
)

for bin_label, r in gp_results.items():
    if r is not None:
        plot_ld_projections(r['X_state_ld'], r['y_state'],
                            EVENT_LABELS, EVENT_COLOURS,
                            f'{mouse_recday} [{bin_label}]')

## Reward × goal-progress LDA

Joint decoding analysis: fits LDA to discriminate reward number AND goal-progress phase. Run across all mouse_recdays and aggregate confusion matrices.

In [ ]:
import lda_reward_goalprogress
reload(lda_reward_goalprogress)
from lda_reward_goalprogress import (
    run_reward_progress_lda_analysis,
    run_reward_progress_decoding,
    plot_aggregate_confusion_matrix,
    plot_decoding_summary,
)

decoding_results = {}
results_by_recday = {}

for mr in mouse_recdays:
    rp_results = run_reward_progress_lda_analysis(
        data_dic, mr,
        valid_sessions=valid_sessions_dic[mr],
        neuron_subset=None,
        min_trials=10,
    )
    if rp_results is None:
        continue
    results_by_recday[mr] = rp_results

    decoding_results[mr] = {}
    for target in ['progress', 'reward']:
        real_acc, null_accs, p_val = run_reward_progress_decoding(
            rp_results, decode_target=target, mouse_recday=mr,
        )
        decoding_results[mr][target] = {
            'real_acc': real_acc,
            'null_accs': null_accs,
            'p_value': p_val,
        }

plot_decoding_summary(decoding_results)
for target in ['progress', 'reward']:
    plot_aggregate_confusion_matrix(results_by_recday, decode_target=target)